# DLAB 10 — Graph learning with hidden liquidity

**Question.** When a sender sees only the *public* channel graph (capacities, fees, topology) and its own
past attempts, can a small graph model predict whether a directed channel can forward a given amount
better than simpler baselines?

**Status.** `Simulation or toy model` — a synthetic, temporal Lightning-like network generated by
`bitct.lngraph`. No live network is probed. A negative or small result is a valid outcome.

**Rule of the game.** Three tables are kept apart:

| table | contains | may a model use it? |
|---|---|---|
| `observations.csv` | capacity, fees, degrees, amount, the observer's own feedback from the previous period | yes |
| `evaluator_only.csv` | the hidden balances | **never** |
| `labels.csv` | `1` if the source side can send `amount` after a 1 % reserve | target only |

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from bitct import lngraph

cfg = lngraph.Config(seed=7)
summary = lngraph.run(cfg, outdir="/lab/lngraph", verbose=False)
obs = pd.read_csv("/lab/lngraph/observations.csv")
hidden = pd.read_csv("/lab/lngraph/evaluator_only.csv")
labels = pd.read_csv("/lab/lngraph/labels.csv")
print(json.load(open("/lab/lngraph/manifest.json"))["tables"])
obs.head()

## 1 · Observable or hidden? (20 min)

Classify every column of `obs`, `hidden` and `labels`: *observable at observation time*, *hidden truth*,
or *label*. Which observable column depends on the past, and why is that still allowed?

In [ ]:
print("observations:", list(obs.columns))
print("evaluator only:", list(hidden.columns))
print("labels:", list(labels.columns))

## 2 · Chronological split and class balance (25 min)

Train on periods 0–7, choose settings on 8–9, test once on 10–11. A temporal split on the same topology
tests *later observations*, not *unseen networks*.

In [ ]:
period = obs["period"].values
y = labels["label"].values
for name, ps in [("train", cfg.train_periods), ("validation", cfg.val_periods), ("test", cfg.test_periods)]:
    m = np.isin(period, ps)
    print(f"{name:<10} rows={m.sum():5d}  periods={list(ps)}  share able to send={y[m].mean():.3f}")

## 3 · Baselines and models on the same test periods (45 min)

* **majority** — always predict the most common training class;
* **capacity heuristic** — score `1 − amount/capacity` (as if the balance were uniformly random);
* **edge-only** logistic regression on this channel's own features;
* **message passing** — each node averages its neighbours' features for *k* hops (k chosen on validation),
  then the same logistic head sees the edge plus both endpoints' neighbourhoods.

In [ ]:
res = summary["results (test periods)"]
table = pd.DataFrame(res).T[["balanced_acc", "pr_auc", "brier"]]
print("hops chosen on validation:", summary["chosen hops (validation)"])
table

**Read the table.** Balanced accuracy ignores the class imbalance; PR-AUC ranks; Brier measures
probability quality (lower is better). Which differences are large enough to matter? One seed is one run.

In [ ]:
o = lngraph.table([r for r in lngraph.simulate(cfg)[0]])
e = labels["label"].values
topo = lngraph.simulate(cfg)[2]
tr, va, te = (np.isin(o["period"], p) for p in (cfg.train_periods, cfg.val_periods, cfg.test_periods))
Xg = lngraph.gnn_features(o, topo, summary["chosen hops (validation)"])
model = lngraph.Logistic().fit(Xg[tr], e[tr])
p = model.predict_proba(Xg[te])
bins = np.linspace(0, 1, 11); idx = np.digitize(p, bins) - 1
centers, freq = [], []
for b in range(10):
    m = idx == b
    if m.sum() > 10:
        centers.append(p[m].mean()); freq.append(e[te][m].mean())
plt.figure(figsize=(4.5, 4)); plt.plot([0, 1], [0, 1], "--", color="gray")
plt.plot(centers, freq, "o-"); plt.xlabel("predicted probability"); plt.ylabel("observed frequency")
plt.title("Calibration of the message-passing model"); plt.show()

## 4 · Leakage diagnosis (30 min)

Two "improvements" below are invalid. Explain *why* each one is leakage and what it would mislead a
reader into believing.

In [ ]:
pd.DataFrame({k: v for k, v in res.items() if k.startswith("LEAKY")}).T

## 5 · One failure case

In [ ]:
wrong = np.where((p > 0.8) & (e[te] == 0))[0]
i = wrong[0] if len(wrong) else int(np.argmax(np.abs(p - e[te])))
row = obs[te].iloc[i]
print("observable:", row[["dir", "amount", "capacity", "fee_ppm", "deg_src", "deg_dst", "prev_success_rate"]].to_dict())
print("model said P(can send) =", round(float(p[i]), 3))
print("hidden truth:", hidden[te].iloc[i][["local_balance", "spendable"]].to_dict())

## 6 · Optional: variability across seeds

In [ ]:
rows = []
for s in range(1, 6):
    r = lngraph.run(lngraph.Config(seed=s), None, verbose=False)["results (test periods)"]
    g = [v for k, v in r.items() if k.startswith("message")][0]
    rows.append({"seed": s, "edge_only_bal_acc": r["edge-only logistic regression"]["balanced_acc"], "message_passing_bal_acc": g["balanced_acc"]})
pd.DataFrame(rows)

## 7 · Conclusion (write it, 25 min)

Report: dataset/seed, permitted features, split boundaries, the metric table, one failure case, and what the
experiment **does not** show (no live Lightning improvement, no routing gain, one simulator, one topology).